# Ejecutar el restaurador de fotos en Kaggle

Alternativa a Google Colab cuando se agota la cuota de GPU gratis.

**Antes de ejecutar (panel derecho, *Session options*):**
1. **Accelerator:** elegí **GPU T4 x2** (la app usa una sola; T4 es la opción más segura).
2. **Internet:** activalo (**On**). Hace falta para clonar el repo y bajar los modelos de Hugging Face. Kaggle exige tener el teléfono verificado para activar GPU e internet.
3. Ejecutá las celdas en orden. Si algo falla, reiniciá la sesión y empezá de nuevo (así no quedan módulos viejos en memoria).
4. Al final se genera un link público temporal (`gradio.live`). No lo compartas ni lo dejes abierto.

In [ ]:
# 1) ¿Hay GPU?
!nvidia-smi

In [ ]:
# 2) ¿Hay internet? Tiene que responder 200 (o 301/302). Si no responde, activá Internet en Session options.
!curl -sI https://huggingface.co | head -1

In [ ]:
# 3) Clonar el repo la primera vez; si ya está, traer los cambios nuevos (se puede re-ejecutar)
RAMA = "feature/subtarea-b"
CARPETA = "imagenes-TI-marketplace-gen"
import os
os.chdir("/kaggle/working")
if os.path.isdir(CARPETA):
    %cd {CARPETA}
    !git pull
else:
    !git clone -b {RAMA} https://github.com/alvarito1709/{CARPETA}.git
    %cd {CARPETA}

In [ ]:
# 4) Instalar dependencias (Kaggle ya trae torch con CUDA, no lo reinstalamos)
!pip install -q gradio diffusers transformers accelerate opencv-python-headless pillow numpy timm einops kornia

In [ ]:
# Prueba rápida del modelo de difusión, sin interfaz (la primera vez descarga ~5 GB)
!python scripts/prueba_difusion.py
from IPython.display import Image as Img, display
display(Img("salidas/prueba_difusion.png"))

In [ ]:
# Evaluación honesta: degrada una foto NÍTIDA, la restaura y mide PSNR/SSIM contra la original.
# Subí una foto buena al repo (por ejemplo assets/ejemplos/foto_buena.jpeg), hacé git push y volvé a correr la celda del clonado.
# Opcional: agregá dos números al final del comando para cambiar el desenfoque y el ruido (por ejemplo: {RUTA} 3 8).
import glob, os
RUTA = "assets/ejemplos/foto_buena.jpeg"
assert os.path.isfile(RUTA), f"No existe {RUTA}: subí una foto nítida ahí (o cambiá RUTA) y hacé git push/pull"
!python scripts/evaluar_degradacion.py {RUTA}
# Cada corrida se guarda en su carpeta; mostramos la más reciente
ultimo = max(glob.glob("salida_evaluacion/*/resultados.md"), key=os.path.getmtime)
print(ultimo)
print(open(ultimo, encoding="utf-8").read())

In [ ]:
# Prueba del detector con una foto de ejemplo (descarga RT-DETR)
!python scripts/prueba_analisis.py assets/ejemplos/foto_cartera.jpeg

In [ ]:
# 6) Lanzar la app. Primero carga los modelos (la primera vez ~4 min) y después
# share=True genera un link público temporal: no lo compartas ni lo dejes abierto.
from app import iniciar
iniciar(share=True)